# Microsoft Agent Framework 실습 노트북
## 고등학생을 위한 AI 에이전트 입문

이 노트북에서는 **Microsoft Agent Framework (MAF)** 를 이용해서
AI 에이전트를 직접 만들어 봅니다.

### 학습 목표
1. AI 에이전트가 무엇인지 코드로 직접 체험한다
2. 에이전트에 **도구(함수)** 를 붙여 능력을 확장한다
3. **여러 에이전트가 협업**하는 모습을 본다
4. 간단한 **워크플로우**를 직접 만들어본다

### 준비물
- Python 3.10 이상
- `pip install agent-framework`
- OpenAI 또는 Azure OpenAI API 키

---
## 0. 환경 준비하기

먼저 필요한 패키지를 설치하고, `.env.sample`을 복사해 APIM 정보를 채웁니다.

 > **TIP:** 키를 코드에 직접 적지 말고 `.env`로 관리하세요.  
 > 노트북에서는 `../.env`를 읽도록 설정합니다.  
 > `cp setup-apim/.env.sample ../.env` 후  
 > `../.env`의 `APIM_BASE_URL`, `APIM_KEY`, `CHAT_MODEL` 값을 채우면 됩니다.

In [15]:
# 패키지 설치 (이미 설치되어 있다면 생략 가능)
# !pip install agent-framework python-dotenv

import os

from dotenv import load_dotenv
from agent_framework.openai import OpenAIChatClient

load_dotenv("../.env", override=True)

def build_chat_client():
    apim_base_url = os.environ["APIM_BASE_URL"].rstrip("/")
    apim_key = os.environ["APIM_KEY"]
    model = os.getenv("CHAT_MODEL", "gpt-5.4")
    return OpenAIChatClient(
        model=model,
        base_url=f"{apim_base_url}/{model}/",
        api_key="placeholder",
        default_headers={"api-key": apim_key},
    )

# APIM 설정이 잘 되었는지 확인
if not os.environ.get("APIM_BASE_URL") or not os.environ.get("APIM_KEY"):
    print("⚠️  APIM 설정이 비어 있어요.")
    print("   1) cp setup-apim/.env.sample ../.env")
    print("   2) ../.env에 APIM_BASE_URL, APIM_KEY를 입력하세요.")
else:
    print("✅ APIM 설정이 준비되었습니다. 시작해 볼까요?")
    print("   CHAT_MODEL =", os.getenv("CHAT_MODEL", "gpt-5.4"))

✅ APIM 설정이 준비되었습니다. 시작해 볼까요?
   CHAT_MODEL = gpt-5.4


---
# 시나리오 1. Hello, Agent! 👋

**목표:** 첫 번째 AI 에이전트를 만들어서 인사를 시켜봅시다.

## 개념 정리
- **Agent**: 대화형 AI 에이전트의 기본 클래스
- **client**: 어떤 LLM(예: GPT-4)을 사용할지 정해주는 부분
- **instructions**: 에이전트의 "성격"이나 "역할"을 정해주는 시스템 메시지
- **name**: 에이전트의 이름 (선택)

In [16]:
from agent_framework import Agent

# 첫 번째 에이전트 만들기
greeter = Agent(
    client=build_chat_client(),
    name="인사봇",
    instructions=(
        "너는 친절한 인사 도우미야. "
        "항상 밝고 따뜻하게, 그리고 한국어로 대답해. "
        "이모지를 적절히 사용해도 좋아."
    ),
)

print("에이전트가 만들어졌어요!")
print(f"이름: {greeter.name}")

에이전트가 만들어졌어요!
이름: 인사봇


이제 에이전트에게 질문해 봅시다.

`agent.run("질문")` 으로 호출하면 답변이 돌아옵니다.
MAF는 `async` 함수이기 때문에 `await` 키워드를 같이 써야 해요.

In [17]:
# 에이전트에게 말 걸기
response = await greeter.run("안녕! 너는 누구니?")
print(response.text)

안녕하세요! 😊  
저는 당신을 도와주는 AI 인사 도우미예요. 궁금한 것에 답해드리고, 대화도 함께할 수 있어요.  

편하게 말 걸어주세요! 🌷


In [18]:
# 한 번 더 — 이번엔 자기소개를 부탁해 봅시다
response = await greeter.run("나는 고등학생이야. AI에 대해 배우고 있어!")
print(response.text)

와, 정말 멋져요! 😊  
고등학생 때 AI를 배우고 있다니 아주 좋은 출발이에요 🚀

AI는 처음엔 어려워 보여도, 차근차근 배우면 정말 재미있는 분야예요.  
예를 들면 이런 것들을 배울 수 있어요:

- **AI가 무엇인지**
- **머신러닝과 딥러닝의 차이**
- **챗봇이 어떻게 대답하는지**
- **이미지나 음성을 AI가 어떻게 인식하는지**
- **AI를 안전하고 올바르게 사용하는 방법**

지금 AI를 공부하고 있다면, 앞으로 프로그래밍, 데이터, 수학과도 자연스럽게 연결될 수 있어서 큰 도움이 될 거예요 📚✨

원하면 내가 이렇게 도와줄 수 있어요:
- AI를 **쉽게 설명해주기**
- 어려운 개념을 **고등학생 눈높이로 정리해주기**
- **발표 준비**나 **보고서 작성** 도와주기
- **AI 관련 진로**도 같이 이야기해주기

궁금한 게 있으면 편하게 물어봐요!  
예를 들면  
**“AI랑 머신러닝은 뭐가 달라?”**  
같이 질문해도 좋아요 😄


### 🤔 생각해보기
- `instructions` 를 바꾸면 에이전트의 말투가 어떻게 달라질까요?
- 예: `"너는 시크한 고양이야. 짧고 도도하게 대답해."` 로 바꿔보세요.

---
# 시나리오 2. 도구를 쓰는 에이전트 🔧

**목표:** AI 에이전트에 **계산기**라는 도구를 붙여서,
직접 계산하지 못하는 LLM이 정확한 답을 낼 수 있게 만들어 봅시다.

## 왜 도구가 필요할까?
GPT 같은 언어 모델은 단어를 예측하는 모델이라서,
**복잡한 계산이나 최신 정보**를 모를 때가 많아요.
이때 "이런 일은 너 대신 이 함수를 써"라고 알려주면
에이전트가 그 함수(=도구)를 직접 호출해서 정답을 가져옵니다.

MAF에서는 그냥 **파이썬 함수**를 정의해서 에이전트에게 넘겨주면 끝!

In [19]:
from typing import Annotated
from pydantic import Field

def add(
    a: Annotated[float, Field(description="첫 번째 숫자")],
    b: Annotated[float, Field(description="두 번째 숫자")],
) -> float:
    """두 숫자를 더한 값을 돌려준다."""
    return a + b

def multiply(
    a: Annotated[float, Field(description="첫 번째 숫자")],
    b: Annotated[float, Field(description="두 번째 숫자")],
) -> float:
    """두 숫자를 곱한 값을 돌려준다."""
    return a * b

def power(
    base: Annotated[float, Field(description="밑")],
    exp: Annotated[float, Field(description="지수")],
) -> float:
    """base의 exp 제곱을 돌려준다."""
    return base ** exp

print("세 가지 도구(add, multiply, power)를 만들었어요.")

세 가지 도구(add, multiply, power)를 만들었어요.


이제 이 함수들을 에이전트에게 **도구**로 등록합니다.

In [23]:
math_helper = Agent(
    client=build_chat_client(),
    name="수학도우미",
    instructions=(
        "너는 수학 문제를 도와주는 친절한 튜터야. "
        "계산이 필요하면 반드시 제공된 도구(add, multiply, power)를 사용해서 답을 구해. "
        "머릿속으로 계산하지 말고 도구를 호출해야 정확해."
    ),
    tools=[add, multiply, power],   # 👈 여기에 함수들을 넘겨요
)

response = await math_helper.run(
    "235 × 47은 얼마야? 그리고 그 결과를 제곱하면?"
)
print(response.text)

235 × 47 = 11045  
그 결과를 제곱하면 11045² = 121992025 입니다.


에이전트가 어떤 도구를 어떻게 호출했는지 살펴볼 수도 있어요.

In [26]:
# 에이전트가 어떤 도구를 어떻게 호출했는지 보기
for msg in response.messages:
    for item in msg.contents:
        if item.type == "function_call":
            print(f"\U0001f527 도구 호출: {item.name}({item.arguments})")
        elif item.type == "function_result":
            print(f"\u2705 도구 결과: {item.result}")
        elif item.type == "text" and item.text.strip():
            print(f"\U0001f4ac 최종 답변: {item.text}")


🔧 도구 호출: multiply({"a":235,"b":47})
✅ 도구 결과: 11045.0
🔧 도구 호출: power({"base":11045,"exp":2})
✅ 도구 결과: 121992025.0
💬 최종 답변: 235 × 47 = 11045  
그 결과를 제곱하면 11045² = 121992025 입니다.


### 🤔 생각해보기
- `divide(a, b)` 도구도 만들어서 추가해 보세요. (0으로 나눌 때 어떻게 처리?)
- 단위 변환 도구(℃ → ℉)를 만들면 에이전트가 "오늘 25도" 같은 질문에 답할 수 있어요.

---
# 시나리오 3. 두 에이전트의 협업 ✍️

**목표:** 서로 다른 역할을 가진 두 AI가 협력해서 글을 쓰는 모습을 봅시다.

## 시나리오
- **작가 에이전트**: 짧은 글을 씁니다
- **편집자 에이전트**: 그 글을 읽고 수정 의견을 줍니다
- 둘이 한 번씩 주고받으며 글의 품질을 높입니다

여러 에이전트를 만들 때는 그냥 `Agent`를 여러 개 만들고
한 에이전트의 결과를 다른 에이전트에게 입력으로 넘기면 됩니다.

In [27]:
# 1. 작가 에이전트
writer = Agent(
    client=build_chat_client(),
    name="작가",
    instructions=(
        "너는 감성적인 단편 작가야. "
        "주제를 받으면 한국어로 4~5문장의 짧은 글을 써. "
        "비유와 묘사를 풍부하게 사용해."
    ),
)

# 2. 편집자 에이전트
editor = Agent(
    client=build_chat_client(),
    name="편집자",
    instructions=(
        "너는 꼼꼼한 편집자야. "
        "받은 글을 읽고, 더 좋아질 수 있는 부분 2~3가지를 짚어줘. "
        "건설적이고 친절한 어조로 피드백해."
    ),
)

print("두 명의 에이전트가 준비됐어요: 작가, 편집자")

두 명의 에이전트가 준비됐어요: 작가, 편집자


In [28]:
# 작가가 먼저 글을 씁니다
topic = "비 오는 날의 등굣길"
draft = await writer.run(f"주제: {topic}. 이 주제로 짧은 글을 써줘.")
print("📝 [작가의 초안]")
print(draft.text)
print()

📝 [작가의 초안]
비 오는 날의 등굣길은 젖은 공책 사이에 끼워 둔 오래된 마음처럼 축축하고도 다정했다.  
우산 위로 떨어지는 빗방울은 작은 북소리처럼 톡톡 울렸고, 회색 하늘 아래 번지는 물웅덩이는 세상을 잠시 거꾸로 비추는 거울이 되었다.  
운동화 끝이 물을 튀길 때마다 차가운 감촉이 발목을 스쳤지만, 이상하게도 그 서늘함마저 아침을 깨우는 인사처럼 느껴졌다.  
교문 앞에 다다랐을 때, 빗속을 한참 걸어온 나는 마치 구름 한 조각을 어깨에 얹고 온 사람처럼 조용히 숨을 골랐다.



In [29]:
# 편집자가 피드백합니다
feedback = await editor.run(
    f"아래는 작가가 쓴 글이야. 어떻게 더 좋아질 수 있을지 의견을 줘.\n\n{draft.text}"
)
print("🧐 [편집자의 피드백]")
print(feedback.text)
print()

🧐 [편집자의 피드백]
글의 분위기가 아주 좋네요. 비 오는 아침의 촉감과 소리를 섬세하게 포착해서, 짧은 글인데도 장면이 선명하게 떠오릅니다. 특히 “젖은 공책 사이에 끼워 둔 오래된 마음”이나 “세상을 잠시 거꾸로 비추는 거울” 같은 표현은 인상적이에요.

더 좋아질 수 있는 지점을 3가지로 짚어보면:

1. **비유의 밀도를 조금만 조절해 보세요.**  
각 문장마다 아름다운 비유가 들어가 있어서 문장 자체는 매우 풍성합니다. 다만 짧은 분량 안에 비유가 연달아 이어지다 보니, 독자가 하나의 이미지를 충분히 음미하기 전에 다음 이미지로 넘어가는 느낌도 있어요.  
예를 들어 몇 군데는 조금 덜 꾸미거나, 핵심 비유 하나를 더 또렷하게 남기면 여운이 더 강해질 수 있습니다. “빗방울은 작은 북소리처럼” 같은 표현은 직관적이고 좋으니, 다른 문장 하나 정도는 조금 담백하게 두어 리듬을 주는 방식도 생각해볼 수 있어요.

2. **화자의 감정이나 상황을 한 발 더 드러내면 몰입이 깊어질 것 같아요.**  
지금 글은 풍경과 감각 묘사가 뛰어나지만, ‘왜 이 아침이 화자에게 다정하게 느껴지는지’가 아주 살짝만 더 보이면 글의 중심이 더 단단해질 수 있습니다.  
예를 들어 단순히 비 오는 등굣길의 정경을 넘어서, 화자가 어떤 마음으로 학교에 가고 있는지—설렘인지, 쓸쓸함인지, 익숙한 위안인지—한 조각만 얹어도 문장이 더 오래 남을 거예요.

3. **마지막 문장을 조금 더 선명하게 마무리해 보세요.**  
“구름 한 조각을 어깨에 얹고 온 사람처럼”이라는 비유는 무척 아름답습니다. 다만 마지막의 “조용히 숨을 골랐다”는 마무리가 안정적이긴 하지만, 앞선 문장들에 비해 다소 익숙하게 느껴질 수도 있어요.  
조금 더 이 글만의 결로 끝내면 좋겠습니다. 예를 들어 빗기운, 젖은 공기, 혹은 학교 앞의 순간과 연결해 마무리하면 처음부터 쌓아온 분위기가 더 응집될 수 있어요.

전체적으로는 이미 문장 감각이 아주 좋은 글이에요. 표현을 덜어낼 곳과 더 드러낼 감정을 조

In [30]:
# 작가가 피드백을 받아 다시 씁니다
final = await writer.run(
    f"네가 쓴 글에 대한 편집자의 피드백이야:\n{feedback.text}\n\n"
    f"이 피드백을 반영해서 글을 다시 써줘."
)
print("✨ [작가의 최종본]")
print(final.text)

✨ [작가의 최종본]
비 오는 아침, 아이들은 우산 끝에 맺힌 물방울을 하나씩 흔들며 학교로 걸어갔다.  
나도 그들 사이에 섞여 걸었지만, 젖은 운동화 속으로 스미는 서늘함이 이상하게 마음을 가라앉혀, 오늘만큼은 늦고 싶은 마음 없이 교문을 향했다.  
빗방울은 처마와 가방 위를 두드리며 작은 북소리를 냈고, 웅덩이마다 잿빛 하늘이 얕게 고여 있었다.  
학교 앞에 다다르자 축축한 바람이 뺨을 스치고 지나갔고, 나는 마치 오래 미뤄 둔 하루를 조용히 펼쳐 드는 사람처럼 숨 대신 빗기운을 한 번 깊게 들이마셨다.


### 🤔 생각해보기
- 편집자에게 "3번 라운드까지 반복하자" 같은 규칙을 줘 보면 어떨까요?
- 사실 검증을 담당하는 **팩트체커 에이전트**를 추가해도 재미있어요.

---
# 시나리오 4. 워크플로우 만들기 🔗

**목표:** 여러 에이전트를 **그래프(워크플로우)** 로 연결해서
자동으로 단계별로 동작하게 만들어 봅시다.

## 워크플로우란?
- 시나리오 3처럼 매번 손으로 결과를 넘겨주는 대신
- 미리 **"누가 → 누구에게 → 어떻게"** 연결할지 정의해 두는 것
- MAF의 `WorkflowBuilder`를 쓰면 시각화도 되고 디버깅도 쉬워요

## 이번 워크플로우
사용자 질문 → **요약 에이전트** (한 줄로 정리) → **번역 에이전트** (영어로 번역) → 결과

> ⚠️ 워크플로우 API는 MAF 버전에 따라 조금씩 달라질 수 있어요.
> 아래는 가장 자주 쓰이는 패턴입니다.

In [32]:
from agent_framework import WorkflowBuilder, AgentExecutor

# 1. 두 개의 에이전트 만들기
summarizer = Agent(
    client=build_chat_client(),
    name="요약가",
    instructions="긴 문장을 한 줄로 핵심만 요약해. 한국어로.",
)

translator = Agent(
    client=build_chat_client(),
    name="번역가",
    instructions="입력 받은 한국어 문장을 자연스러운 영어로 번역해.",
)

# 2. 워크플로우 만들기
summarize_step = AgentExecutor(summarizer, id="summarize")
translate_step = AgentExecutor(translator, id="translate")

workflow = (
    WorkflowBuilder(start_executor=summarize_step)
    .add_edge(summarize_step, translate_step)
    .build()
)

print("워크플로우가 만들어졌어요!")
print("흐름: 입력 \u2192 요약가 \u2192 번역가 \u2192 출력")


워크플로우가 만들어졌어요!
흐름: 입력 → 요약가 → 번역가 → 출력


/tmp/ipykernel_24288/273541010.py:23: DeprecationWarning: WorkflowBuilder built without explicit output_from or intermediate_output_from; every yield_output produces type='output' for compatibility. Pass output_from='all', output_from=[...], or intermediate_output_from=[...] to opt into explicit designation - explicit designation will be required in a future version.
  .build()


In [33]:
# 3. 워크플로우 실행
long_text = (
    "오늘 학교에서 과학 수업 시간에 인공지능에 대해 배웠다. "
    "선생님께서는 AI가 단순히 챗봇이 아니라 스스로 도구를 사용하고 "
    "여러 단계를 거쳐 일을 해내는 에이전트로 발전하고 있다고 하셨다. "
    "수업이 끝나고 친구들과 함께 다음 주에 직접 만들어 보기로 했다."
)

events = await workflow.run(long_text)

# 결과 출력 (get_outputs()는 리스트를 돌려줘요)
print("\U0001f4e5 [입력]")
print(long_text)
print()
print("\U0001f4e4 [최종 결과]")
for output in events.get_outputs():
    print(output)


📥 [입력]
오늘 학교에서 과학 수업 시간에 인공지능에 대해 배웠다. 선생님께서는 AI가 단순히 챗봇이 아니라 스스로 도구를 사용하고 여러 단계를 거쳐 일을 해내는 에이전트로 발전하고 있다고 하셨다. 수업이 끝나고 친구들과 함께 다음 주에 직접 만들어 보기로 했다.

📤 [최종 결과]
학교 과학 시간에 AI가 여러 단계를 거쳐 도구를 사용하는 에이전트로 발전하고 있음을 배우고, 다음 주 친구들과 직접 만들어 보기로 했다.
Today in science class at school, I learned about artificial intelligence. Our teacher said that AI is evolving beyond simple chatbots into agents that can use tools on their own and accomplish tasks through multiple steps. After class, my friends and I decided to try making one ourselves next week.


### 🤔 생각해보기
- 번역 다음에 **분위기 분석 에이전트**를 한 단계 더 추가해 보세요.
- 어떤 조건일 때만 한쪽 길로 가는 **분기(branch)** 도 만들 수 있어요.

---
# 🎓 마무리 & 다음 단계

오늘 우리는:

1. **첫 에이전트** 를 만들어 인사를 시켜 봤고  
2. **도구(함수)** 를 붙여 계산을 정확하게 시켰고  
3. **두 에이전트** 가 작가-편집자로 협업하게 했고  
4. **워크플로우** 로 자동 파이프라인을 만들었습니다.

## 응용 아이디어 💡
- **나만의 학습 도우미**: 단어 퀴즈 에이전트 + 채점 에이전트
- **여행 플래너**: 검색 에이전트 + 일정 정리 에이전트 + 예산 계산 에이전트
- **창작 도우미**: 시놉시스 작가 + 등장인물 디자이너 + 일러스트 프롬프트 작성가
- **뉴스 브리핑**: 기사 수집기 + 요약가 + 음성 변환기

## 더 알아보기 📚
- 공식 문서: <https://learn.microsoft.com/agent-framework/>
- GitHub: <https://github.com/microsoft/agent-framework>
- Python 패키지: `pip install agent-framework`

**기억하세요:** 좋은 AI 에이전트는 화려한 코드가 아니라
**좋은 instructions**(역할/규칙)과 **잘 설계된 도구**에서 나옵니다.  
오늘 배운 걸로 여러분만의 작은 AI를 만들어 보세요! 🚀